# Embedding

We start by getting some text

In [ ]:
with open("frankenstein.md", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:200])

If you do not have the `frankenstein.txt` file, you can download directly from Gutenberg project with the following code. It is optional, use it only if the previous code fails.

In [ ]:
import urllib.request

with urllib.request.urlopen('https://gutenberg.org/ebooks/84.txt.utf-8') as response:
   text = response.read()

print(text[:200].decode('utf-8'))

We need to separate the text into tokens. For this exercise we will take each word as a token. Let's make a function producing tokens.

In [ ]:
def tokenize(text: str) -> list[str]:
  ...

In [ ]:
tokens = tokenize(text)
print(len(tokens), tokens[:10])

In [ ]:
tokens = tokens[:500] 

In [ ]:
def mapping(tokens: list[str]) -> tuple[dict[str, int], dict[int, str]]:
    word_to_id = {}
    id_to_word = {}
    
    for i, token in enumerate(set(tokens)):
        word_to_id[token] = i
        id_to_word[i] = token
    
    return word_to_id, id_to_word

In [ ]:
word_to_id, id_to_word = mapping(tokens)
word_to_id

We need to see the context of  each word

![](https://miro.medium.com/max/1400/1*Mmp1vbFOxrmiCF17lYJWRA.png)

that we will represent as

```py
["back-alleys", "little"]
["back-alleys", "dark"]
["back-alleys", "behind"]
["back-alleys", "the"]
```

Moreover, we want to encode the words as `one_hot`

In [ ]:
from typing import Iterable

def concat(*iterables: Iterable) -> Iterable:
    for iterable in iterables:
        yield from iterable

def generate_training_data(
        tokens: list[str],
        window: int) -> tuple[list[str], list[str]]:

    wrd = []
    cntxt = []
    n_tokens = len(tokens)
    
    for i in range(n_tokens):
        idx = concat(
            range(max(0, i - window), i), 
            range(i, min(n_tokens, i + window + 1))
        )
        for j in idx:
            if i == j:
                continue
            wrd.append(tokens[i])
            cntxt.append(tokens[j])
    
    return wrd, cntxt

In [ ]:
import numpy as np

def one_hot_encode(index: int, vocab_size: int) -> np.ndarray:
    one_hot = np.zeros(vocab_size)
    one_hot[index] = 1
    return one_hot

def encode_training_data(
        wrd: list[str],
        word_to_id: dict[str, int]
        ) -> np.ndarray:

    enc = []
    
    for w in wrd:
      enc.append(one_hot_encode(word_to_id[w], len(word_to_id)))
    
    return np.asarray(enc)

In [ ]:
words, context = generate_training_data(tokens, 2)
X = encode_training_data(words, word_to_id)
y = encode_training_data(context, word_to_id)
print(X.shape, y.shape)

In [ ]:
class Model:
    def __init__(self, vocab_size: int, n_embedding: int):
        self.w1 = np.random.randn(vocab_size, n_embedding)
        self.w2 = np.random.randn(n_embedding, vocab_size)

In [ ]:
model = Model(len(word_to_id), 10)
model.w1.shape, model.w2.shape

Sometimes it is useful to visualize the matrices with a heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(model.w1, cmap="viridis")
plt.show()

In [ ]:
def softmax(X: np.ndarray) -> np.ndarray:
    e = np.exp(X)
    s = e / np.sum(e, axis=1, keepdims=True)
    return s

In [ ]:
class FwData:
    a1: np.ndarray
    a2: np.ndarray
    z: np.ndarray


def forward(model: Model, X: np.ndarray) -> FwData:
    fw_data = FwData()

    fw_data.a1 = X @ model.w1
    fw_data.a2 = fw_data.a1 @ model.w2
    fw_data.z = softmax(fw_data.a2)
    
    return fw_data

In [ ]:
def cross_entropy(z, y):
    return - np.sum(np.log(z) * y)

In [ ]:
def backward(model, X, y, alpha):
    fw_data  = forward(model, X)
    da2 = fw_data.z - y
    dw2 = fw_data.a1.T @ da2
    da1 = da2 @ model.w2.T
    dw1 = X.T @ da1
    assert(dw2.shape == model.w2.shape)
    assert(dw1.shape == model.w1.shape)
    model.w1 -= alpha * dw1
    model.w2 -= alpha * dw2
    return cross_entropy(fw_data.z, y)

In [ ]:
n_iter = 5
learning_rate = 0.005

history = [backward(model, X, y, learning_rate) for _ in range(n_iter)]


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

plt.plot(range(len(history)), history, color="skyblue")
plt.show()